# Phase 1 — Exploratory Data Analysis

Dataset: blood cell images — 4 classes: **Eosinophil, Lymphocyte, Monocyte, Neutrophil**

Before we build anything, let's just look at what we have. We want to check:
- how many images per class (is it balanced?)
- what the cells actually look like
- whether image sizes are consistent
- how pixel intensities differ between classes
- what our augmentation transforms look like on real images

All figures are saved to `outputs/figures/`.

In [ ]:
import os
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

plt.rcParams['figure.dpi']       = 110
plt.rcParams['axes.spines.top']  = False
plt.rcParams['axes.spines.right']= False
plt.rcParams['font.size']        = 10

print('imports ok')

In [ ]:
# if running from notebooks/, step up to the project root
if Path.cwd().name == 'notebooks':
    os.chdir('..')

ROOT       = Path('.')
DATA_ROOT  = Path('../Data')
TRAIN_DIR  = DATA_ROOT / 'TRAIN'
TEST_DIR   = DATA_ROOT / 'TEST'
SIMPLE_DIR = DATA_ROOT / 'TEST_SIMPLE'
FIGS       = ROOT / 'outputs' / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)

CLASSES = ['EOSINOPHIL', 'LYMPHOCYTE', 'MONOCYTE', 'NEUTROPHIL']
COLORS  = ['#4FA3D1', '#5DBB8A', '#E8A838', '#D15F4F']

for p in [TRAIN_DIR, TEST_DIR, SIMPLE_DIR]:
    status = 'OK' if p.exists() else 'MISSING'
    print(f'  {status}  {p}')

## 1. How many images do we have?

Count images per class in each split.

In [ ]:
def count_cls(split_dir):
    result = {}
    for cls in CLASSES:
        files = [f for f in (split_dir / cls).glob('*')
                 if f.suffix.lower() in ('.jpg', '.jpeg', '.png')]
        result[cls] = len(files)
    return result

splits = {'TRAIN': TRAIN_DIR, 'TEST': TEST_DIR, 'TEST_SIMPLE': SIMPLE_DIR}
counts = {s: count_cls(d) for s, d in splits.items()}

# print table
print(f"{'Class':<16}", end='')
for s in splits:
    print(f"{s:>12}", end='')
print()
print('-' * 52)
for cls in CLASSES:
    print(f"{cls:<16}", end='')
    for s in splits:
        print(f"{counts[s][cls]:>12}", end='')
    print()
print('-' * 52)
print(f"{'TOTAL':<16}", end='')
for s in splits:
    print(f"{sum(counts[s].values()):>12}", end='')
print()

## 2. What do the cells actually look like?

8 random images from each class.

In [ ]:
fig, axes = plt.subplots(4, 8, figsize=(16, 9))

for row, (cls, color) in enumerate(zip(CLASSES, COLORS)):
    paths  = list((TRAIN_DIR / cls).glob('*.jpeg'))
    chosen = random.sample(paths, 8)
    for col, p in enumerate(chosen):
        ax = axes[row, col]
        ax.imshow(np.array(Image.open(p)))
        ax.axis('off')
    # class label on the left side
    axes[row, 0].set_ylabel(cls, fontsize=9, color=color,
                             rotation=0, labelpad=72, va='center')

fig.suptitle('8 random samples per class (TRAIN)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIGS / 'eda_01_sample_grid.png', bbox_inches='tight', dpi=150)
plt.show()
print('saved -> outputs/figures/eda_01_sample_grid.png')

## 3. Is the dataset balanced?

A balanced dataset means we don't need class weighting or oversampling tricks.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (split, _) in zip(axes, splits.items()):
    vals = [counts[split][cls] for cls in CLASSES]
    bars = ax.bar(CLASSES, vals, color=COLORS, width=0.55, edgecolor='white', linewidth=0.8)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(vals) * 0.012,
                str(v), ha='center', va='bottom', fontsize=9)
    ax.set_title(split, fontsize=11, fontweight='bold')
    ax.set_ylim(0, max(vals) * 1.20)
    ax.set_xticklabels(CLASSES, rotation=20, ha='right', fontsize=8)
    ax.set_ylabel('image count')

fig.suptitle('Class distribution across all splits', fontsize=13)
plt.tight_layout()
plt.savefig(FIGS / 'eda_02_class_distribution.png', bbox_inches='tight', dpi=150)
plt.show()
print('saved -> outputs/figures/eda_02_class_distribution.png')

## 4. Image dimensions

Quick check — are all images the same size? We need to know what we're resizing from.

In [ ]:
# sample 300 images — enough to catch any size variation without loading everything
all_paths = list(TRAIN_DIR.glob('*/*.jpeg'))
sample    = random.sample(all_paths, min(300, len(all_paths)))
sizes     = [Image.open(p).size for p in sample]   # (width, height)

size_counts = Counter(sizes)
widths  = [s[0] for s in sizes]
heights = [s[1] for s in sizes]

print(f'Unique sizes found : {len(size_counts)}')
print(f'Most common        : {size_counts.most_common(3)}')
print(f'Width  — min {min(widths)}, max {max(widths)}, mean {np.mean(widths):.0f}')
print(f'Height — min {min(heights)}, max {max(heights)}, mean {np.mean(heights):.0f}')
print(f'All same size      : {len(size_counts) == 1}')
print(f'\nAll images will be resized to 224×224 for training.')

## 5. Pixel intensity per class

Different cell types are stained differently under the microscope.
Eosinophils are pink/orange, neutrophils are more purple — this should show up in the R/G/B distributions.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for ax, (cls, color) in zip(axes, zip(CLASSES, COLORS)):
    # 80 images per class is plenty for the histogram
    paths = random.sample(list((TRAIN_DIR / cls).glob('*.jpeg')), 80)
    r_all, g_all, b_all = [], [], []
    for p in paths:
        arr = np.array(Image.open(p))
        r_all.append(arr[:, :, 0].ravel())
        g_all.append(arr[:, :, 1].ravel())
        b_all.append(arr[:, :, 2].ravel())

    bins = np.linspace(0, 255, 60)
    ax.hist(np.concatenate(r_all), bins=bins, color='#E05252',
            alpha=0.55, density=True, label='R')
    ax.hist(np.concatenate(g_all), bins=bins, color='#52A852',
            alpha=0.55, density=True, label='G')
    ax.hist(np.concatenate(b_all), bins=bins, color='#5278E0',
            alpha=0.55, density=True, label='B')
    ax.set_title(cls, fontsize=11, color=color, fontweight='bold')
    ax.set_xlabel('pixel value (0–255)')
    ax.set_ylabel('density')
    ax.legend(fontsize=8)

fig.suptitle('RGB channel distributions — 80 images per class', fontsize=13)
plt.tight_layout()
plt.savefig(FIGS / 'eda_03_pixel_histograms.png', bbox_inches='tight', dpi=150)
plt.show()
print('saved -> outputs/figures/eda_03_pixel_histograms.png')

## 6. Mean image per class

Average 100 images per class to see the typical cell shape.
A sharp mean = consistent cell shape. A blurry mean = high variation within that class.

In [ ]:
TARGET = (320, 240)   # keep original resolution for the mean image

fig, axes = plt.subplots(1, 4, figsize=(15, 4))

for ax, (cls, color) in zip(axes, zip(CLASSES, COLORS)):
    paths = random.sample(list((TRAIN_DIR / cls).glob('*.jpeg')), 100)
    stack = np.stack(
        [np.array(Image.open(p).resize(TARGET)) for p in paths]
    )   # shape: (100, 240, 320, 3)
    mean_img = stack.mean(axis=0).astype(np.uint8)
    ax.imshow(mean_img)
    ax.set_title(cls, fontsize=10, color=color, fontweight='bold')
    ax.axis('off')

fig.suptitle('Mean image per class  (average of 100 samples)', fontsize=13)
plt.tight_layout()
plt.savefig(FIGS / 'eda_04_mean_images.png', bbox_inches='tight', dpi=150)
plt.show()
print('saved -> outputs/figures/eda_04_mean_images.png')

## 7. What our augmentation looks like

We apply these transforms to training images only — never to val or test.
The idea is to show the model each cell in many different orientations and lighting conditions, so it doesn't memorise the exact training images.

Transforms used: rotation ±15°, horizontal + vertical flip, zoom ±10%, width/height shift ±10%, brightness ±20%.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

aug = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.10,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.80, 1.20],
    fill_mode='nearest'
)

fig, axes = plt.subplots(4, 7, figsize=(15, 10))

for row, (cls, color) in enumerate(zip(CLASSES, COLORS)):
    img_path = random.choice(list((TRAIN_DIR / cls).glob('*.jpeg')))
    img = np.array(Image.open(img_path).resize((224, 224)))

    # column 0 — original
    axes[row, 0].imshow(img)
    axes[row, 0].axis('off')
    axes[row, 0].set_ylabel(cls, fontsize=9, color=color,
                             rotation=0, labelpad=72, va='center')

    # columns 1–6 — 6 random augmented versions
    gen = aug.flow(img[np.newaxis], batch_size=1)
    for col in range(1, 7):
        aug_img = next(gen)[0].clip(0, 255).astype(np.uint8)
        axes[row, col].imshow(aug_img)
        axes[row, col].axis('off')

# titles on the top row only
col_titles = ['original', 'aug 1', 'aug 2', 'aug 3', 'aug 4', 'aug 5', 'aug 6']
for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=9)

fig.suptitle('Original vs 6 random augmentations — one image per class', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIGS / 'eda_05_augmentation_preview.png', bbox_inches='tight', dpi=150)
plt.show()
print('saved -> outputs/figures/eda_05_augmentation_preview.png')

## Summary

Here's what we found:

| | |
|---|---|
| **Total images** | 9,958 train / 2,488 test / 72 TEST_SIMPLE |
| **Balance** | All 4 classes nearly equal — no weighting needed |
| **Image size** | 320 × 240 px, RGB JPEG — consistent across all splits |
| **Visual differences** | Each class has a clearly distinct look under the microscope |
| **Pixel stats** | Channel distributions differ between classes — staining works as expected |
| **Mean images** | Each class averages to a recognisable shape |
| **Augmentation** | Transforms add variety without making cells unrecognisable |

The dataset looks clean and ready for training. Moving on to Phase 2 — custom CNN.